## Imports

In [6]:
import os
import re
import mne
import pathlib
import shutil
from tqdm import tqdm
import pandas as pd
from pathlib import Path

## Functions

In [2]:
def get_subject_folders(parent_directory):
    """
    Returns a list of paths for all subfolders 
    starting with 'sub-' in the given directory.
    """
    path = Path(parent_directory)
    
    # .glob('sub-*') looks for items starting with 'sub-'
    # is_dir() ensures we only get folders, not files
    subject_folders = [str(f) for f in path.glob('sub-*') if f.is_dir()]
    
    return sorted(subject_folders)

# Example Usage:
# folders = get_subject_folders('/path/to/your/eeg_data')
# print(f"Found {len(folders)} subject folders.")

In [3]:
def extract_unique_stimuli(file_path):
    """
    Parses a BrainVision .vmrk file and returns a sorted list 
    of all unique stimulus descriptions (trigger codes).
    """
    stimuli = set()
    
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            for line in file:
                # Look for lines starting with Mk followed by digits (e.g., Mk100=...)
                if line.startswith('Mk'):
                    # Split by '=' to separate key and values, then split values by ','
                    try:
                        # Format: Mk<Num>=Type,Description,Position,Size,Channel
                        content = line.split('=')[1]
                        parts = content.split(',')
                        
                        marker_type = parts[0].strip()
                        description = parts[1].strip()
                        
                        # Only add if the type is exactly 'Stimulus'
                        if marker_type == 'Stimulus':
                            stimuli.add(description)
                    except (IndexError, ValueError):
                        # Skip malformed lines
                        continue
                        
    except FileNotFoundError:
        return "Error: File not found."

    # Return as a sorted list for easier viewing
    return sorted(list(stimuli))

## Variables

## Main

In [4]:
main_data_folder = "/Users/jonasadomaitis/Masters/FDA/Functional-Data-Analysis/ds006018"
tasks = ["task-auditoryoddball", "task-flanker", "task-visualoddball", "task-visualsearch"]

In [5]:
paths_to_original_data_actors = get_subject_folders(main_data_folder)

In [9]:
for i in tqdm(range(len(paths_to_original_data_actors))):

    for task_no in range(len(tasks)):

        task = tasks[task_no]

        sub_number = paths_to_original_data_actors[i].split('/')[-1]

        path_to_vhdr = paths_to_original_data_actors[i] + "/eeg/" + sub_number + "_" + task + "_eeg.vhdr"
        path_to_vmrk = paths_to_original_data_actors[i] + "/eeg/" + sub_number + "_" + task + "_eeg.vmrk"

        directory = Path("./ds006018_per_stimuli/"+sub_number)
        directory.mkdir(parents=True, exist_ok=True)

        file_path = Path(path_to_vhdr)

        if file_path.is_file():
            print("The file exists!")
        


            # Get unique stimulus numbers from the .vmrk file
            unique_stimuli_numers = extract_unique_stimuli(path_to_vmrk)
            print(unique_stimuli_numers)


            # 1. Load the BrainVision data
            raw = mne.io.read_raw_brainvision(path_to_vhdr, preload=True)


            # 2. Set the montage (Standard 10-20 system for electrode locations)
            montage = mne.channels.make_standard_montage('standard_1020')
            raw.set_montage(montage)


            # 3. Filtering (Standard for EEG: 0.1Hz to 40Hz)
            raw.filter(l_freq=0.1, h_freq=40.0)


            # 4. Plotting the data to inspect for noise
            #raw.plot(n_channels=15, duration=5, scalings='auto')


            # 5. Preprocessing (Required for FDA to reduce noise)
            #raw.resample(200)               # Downsample to reduce R processing time - let's do this since we have a lot of data and FDA can be computationally intensive

            # 6. Create Epochs (FDA usually analyzes trials/segments) - by 
            # This assumes you have event markers in your .vmrk file
            events, event_id = mne.events_from_annotations(raw)

            stimulus_dataframes = {}

            # Iterate through every stimulus
            for event_name, event_val in event_id.items():
                # Clean the name for filenames (e.g., 'Stimulus_S1')
                clean_name = event_name.replace('/', '_').replace(' ', '')
                
                try:
                    # Pass a dictionary where the key is the name and value is the integer ID
                    # This resolves the "must be an int, got str" error
                    current_epochs = mne.Epochs(raw, events, event_id={event_name: event_val}, 
                                                tmin=-0.2, tmax=0.8, preload=True)
                    
                    if len(current_epochs) > 0:
                        df_temp = current_epochs.to_data_frame()
                        
                        # Store and export
                        stimulus_dataframes[clean_name] = df_temp
                        df_temp.to_csv("./ds006018_per_stimuli/"+sub_number+"/"+task+"_"+clean_name+".csv", index=False)
                        
                        print(f"Successfully created: eeg_{clean_name}.csv ({len(current_epochs)} trials)")
                        
                except Exception as e:
                    print(f"Skipping {event_name}: {e}")

        else:
            print(path_to_vhdr+"File not found.")

  0%|          | 0/127 [00:00<?, ?it/s]

The file exists!
['S  1', 'S 70', 'S 80', 'S180']
Extracting parameters from /Users/jonasadomaitis/Masters/FDA/Functional-Data-Analysis/ds006018/sub-001/eeg/sub-001_task-auditoryoddball_eeg.vhdr...
Setting channel info structure...
Reading 0 ... 124029  =      0.000 ...   248.058 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.10
- Lower transition bandwidth: 0.10 Hz (-6 dB cutoff frequency: 0.05 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 16501 samples (33.002 s)

Used Annotations descriptions: [np.str_('Stimulus/S  1'), np.str_('Stimulus/S 70'), np.str_('Stimulus/S 80'), np.str_('Stimulus/S180')]
Not setti

/var/folders/h_/1807vbxj1l7cgpryjss3n4c80000gn/T/ipykernel_67640/432237245.py:28: RuntimeWarning: No coordinate information found for channels ['HEL', 'LM', 'RM', 'HER', 'VER']. Setting channel types to misc. To avoid this warning, set channel types explicitly.
  raw = mne.io.read_raw_brainvision(path_to_vhdr, preload=True)
/var/folders/h_/1807vbxj1l7cgpryjss3n4c80000gn/T/ipykernel_67640/432237245.py:28: RuntimeWarning: Not setting positions of 5 misc channels found in montage:
['HEL', 'LM', 'RM', 'HER', 'VER']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_brainvision(path_to_vhdr, preload=True)


0 bad epochs dropped
Successfully created: eeg_Stimulus_S70.csv (70 trials)
Not setting metadata
265 matching events found
Setting baseline interval to [-0.2, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 265 events and 501 original time points ...
0 bad epochs dropped
Successfully created: eeg_Stimulus_S80.csv (265 trials)
Not setting metadata
15 matching events found
Setting baseline interval to [-0.2, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 15 events and 501 original time points ...
0 bad epochs dropped
Successfully created: eeg_Stimulus_S180.csv (15 trials)
The file exists!
['S  2', 'S 11', 'S 12', 'S 21', 'S 22', 'S111', 'S112', 'S121', 'S122', 'S211', 'S212', 'S221', 'S222']
Extracting parameters from /Users/jonasadomaitis/Masters/FDA/Functional-Data-Analysis/ds006018/sub-001/eeg/sub-001_task-flanker_eeg.vhdr...
Setting channel info structure..

/var/folders/h_/1807vbxj1l7cgpryjss3n4c80000gn/T/ipykernel_67640/432237245.py:28: RuntimeWarning: No coordinate information found for channels ['HEL', 'LM', 'RM', 'HER', 'VER']. Setting channel types to misc. To avoid this warning, set channel types explicitly.
  raw = mne.io.read_raw_brainvision(path_to_vhdr, preload=True)
/var/folders/h_/1807vbxj1l7cgpryjss3n4c80000gn/T/ipykernel_67640/432237245.py:28: RuntimeWarning: Not setting positions of 5 misc channels found in montage:
['HEL', 'LM', 'RM', 'HER', 'VER']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_brainvision(path_to_vhdr, preload=True)


...

Successfully created: eeg_Stimulus_S70.csv (70 trials)
Not setting metadata
265 matching events found
Setting baseline interval to [-0.2, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 265 events and 501 original time points ...
0 bad epochs dropped
Successfully created: eeg_Stimulus_S80.csv (265 trials)
Not setting metadata
15 matching events found
Setting baseline interval to [-0.2, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 15 events and 501 original time points ...
0 bad epochs dropped
Successfully created: eeg_Stimulus_S180.csv (15 trials)
/Users/jonasadomaitis/Masters/FDA/Functional-Data-Analysis/ds006018/sub-127/eeg/sub-127_task-flanker_eeg.vhdrFile not found.
The file exists!
['S 11', 'S 12', 'S 13', 'S 14', 'S 15', 'S 21', 'S 22', 'S 23', 'S 24', 'S 25', 'S 31', 'S 32', 'S 33', 'S 34', 'S 35', 'S 41', 'S 42', 'S 43', 'S 44', 'S 45', 'S 51', 'S 5

/var/folders/h_/1807vbxj1l7cgpryjss3n4c80000gn/T/ipykernel_67640/432237245.py:28: RuntimeWarning: No coordinate information found for channels ['HEL', 'LM', 'RM', 'HER', 'VER']. Setting channel types to misc. To avoid this warning, set channel types explicitly.
  raw = mne.io.read_raw_brainvision(path_to_vhdr, preload=True)
/var/folders/h_/1807vbxj1l7cgpryjss3n4c80000gn/T/ipykernel_67640/432237245.py:28: RuntimeWarning: Not setting positions of 5 misc channels found in montage:
['HEL', 'LM', 'RM', 'HER', 'VER']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_brainvision(path_to_vhdr, preload=True)


Not setting metadata
4 matching events found
Setting baseline interval to [-0.2, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 4 events and 501 original time points ...
0 bad epochs dropped
Successfully created: eeg_Comment_BufferOverflow.csv (4 trials)
Not setting metadata
4 matching events found
Setting baseline interval to [-0.2, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 4 events and 501 original time points ...
0 bad epochs dropped
Successfully created: eeg_NewSegment_.csv (4 trials)
Not setting metadata
8 matching events found
Setting baseline interval to [-0.2, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 8 events and 501 original time points ...
0 bad epochs dropped
Successfully created: eeg_Stimulus_S11.csv (8 trials)
Not setting metadata
8 matching events found
Setting baseline

100%|██████████| 127/127 [40:03<00:00, 18.93s/it]

Successfully created: eeg_Stimulus_S202.csv (29 trials)
/Users/jonasadomaitis/Masters/FDA/Functional-Data-Analysis/ds006018/sub-127/eeg/sub-127_task-visualsearch_eeg.vhdrFile not found.
